# 02 - Fine-Tuning IndoT5 dengan Grid Search Hyperparameter

Total eksperimen: **1 baseline + 18 grid search = 19 run**

| # | Konfigurasi | Epoch | Batch Size | Learning Rate |
|---|---|---|---|---|
| 0 | **Baseline** (default, tanpa tuning) | - | - | - |
| 1–18 | **Grid Search** | 5, 10, 15 | 4, 8 | 1e-4, 3e-5, 5e-5 |

> Setiap run dievaluasi dengan ROUGE pada `test.csv`. Model dengan ROUGE-L tertinggi disimpan sebagai model final.

##Login Hugging Face

In [1]:
from huggingface_hub import login
from google.colab import userdata

try:
    my_token = userdata.get('HF_TOKEN')
    if my_token:
        login(token=my_token, add_to_git_credential=True)
        print("Berhasil login ke Hugging Face!")
    else:
        print("Token HF_TOKEN tidak ditemukan di Secrets Colab.")
except Exception as e:
    print(f"Gagal mengambil secret: {e}")

Berhasil login ke Hugging Face!


##Mount Drive & Install

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers datasets evaluate sentencepiece accelerate rouge-score -q

import shutil

DRIVE_DATA_DIR = '/content/drive/MyDrive/data_latih'

shutil.copy(f'{DRIVE_DATA_DIR}/train.csv', 'train.csv')
shutil.copy(f'{DRIVE_DATA_DIR}/test.csv',  'test.csv')
print("Data berhasil di-copy dari Google Drive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data berhasil di-copy dari Google Drive!


## 3. Import & Konfigurasi

In [3]:
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
from datasets import Dataset
import pandas as pd
import evaluate
import torch
from pathlib import Path

MODEL_NAME  = "Wikidepia/IndoT5-base"
OUTPUT_ROOT = Path('/content/drive/MyDrive/models')
BEST_DIR    = OUTPUT_ROOT / 'indot5_finetuned'
LOG_DIR     = OUTPUT_ROOT / 'training_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

PREFIX  = 'ringkas: '
MAX_IN  = 512
MAX_OUT = 128

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'Model  : {MODEL_NAME}')

Device : cuda
Model  : Wikidepia/IndoT5-base


## 4. Data Loading & Tokenization

In [12]:
df_train = pd.read_csv('train.csv')
df_test  = pd.read_csv('test.csv')
print(f'Train: {len(df_train)} | Test: {len(df_test)}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenisasi(examples):
    inputs = [PREFIX + str(s) for s in examples['input_whisper_segment']]
    m_in   = tokenizer(inputs, max_length=MAX_IN, truncation=True)

    labels = tokenizer(
        text_target=[str(t) for t in examples['target_summary_manual']],
        max_length=MAX_OUT, truncation=True
    )

    # Pad token di label diganti -100 supaya tidak ikut dihitung dalam loss
    m_in['labels'] = [
        [-100 if t == tokenizer.pad_token_id else t for t in seq]
        for seq in labels['input_ids']
    ]
    return m_in

train_ds = Dataset.from_pandas(df_train).map(
    tokenisasi, batched=True, remove_columns=df_train.columns.tolist()
)
test_ds = Dataset.from_pandas(df_test).map(
    tokenisasi, batched=True, remove_columns=df_test.columns.tolist()
)

# Debug: pastikan label sudah benar sebelum training
sample = train_ds[0]
labels_non_pad = [t for t in sample['labels'] if t != -100]
print(f'\nCek label[0]:')
print(f'  Total token : {len(sample["labels"])}')
print(f'  Non-pad (-100 diganti): {len(labels_non_pad)}')
print(f'  Ada -100 mask : {-100 in sample["labels"]}')
# Harusnya: Ada -100 mask = True, Non-pad > 0
# Kalau Non-pad = 0 → semua label jadi -100 → loss NaN

print('\nTokenisasi selesai.')

Train: 390 | Test: 96


Map:   0%|          | 0/390 [00:00<?, ? examples/s]

Map:   0%|          | 0/96 [00:00<?, ? examples/s]


Cek label[0]:
  Total token : 65
  Non-pad (-100 diganti): 65
  Ada -100 mask : False

Tokenisasi selesai.


## 5. Diagnostik (Opsional) Lihat Baseline

In [5]:
import gc

model_test     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
tokenizer_test = AutoTokenizer.from_pretrained(MODEL_NAME)

sample_input = PREFIX + str(df_test['input_whisper_segment'].iloc[0])[:300]
inputs  = tokenizer_test(sample_input, return_tensors='pt', max_length=512, truncation=True).to(DEVICE)
outputs = model_test.generate(**inputs, max_new_tokens=100)

print("PREFIX yang dipakai    :", repr(PREFIX))
print("Input awal (Whisper)   :", sample_input[:100], "...")
print("Output model mentah    :", tokenizer_test.decode(outputs[0], skip_special_tokens=True))
print("Referensi (Manual)     :", str(df_test['target_summary_manual'].iloc[0])[:200], "...")

del model_test, tokenizer_test
torch.cuda.empty_cache()
gc.collect()
print("VRAM diagnostik sudah dibersihkan.")

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

PREFIX yang dipakai    : 'ringkas: '
Input awal (Whisper)   : ringkas: kesadaran kami persilakan untuk duduk kembali sidang dewan yang kami hormati sesuai dengan  ...
Output model mentah    : ini adalah: ini adalah: kami hormati, kami hormati, kami hormati, kami hormati, kami hormati, kami hormati, kami hormati, ini adalah: ini adalah ini adalah: ini adalah ini adalah ini adalah rapat ini adalah ini adalah adalah ini adalah rapat ini adalah ini, ini ini ini24, ini adalah ini adalahs purn rapat komisi 3
Referensi (Manual)     : Rapat Paripurna DPR RI membahas laporan Komisi III tentang hasil uji kelayakan calon anggota Lembaga Perlindungan Saksi dan Korban masa jabatan 2024-2029. Pimpinan Komisi III, Dr. Habibur Rahman, meny ...
VRAM diagnostik sudah dibersihkan.


## 6. CELL 6 — Definisi Fungsi Helper (hitung_rouge & jalankan_satu_run)

In [13]:
import gc
import inspect
from torch.utils.data import DataLoader
from transformers import DataCollatorForSeq2Seq

LOCAL_BEST_DIR = Path('/content/model_juara_lokal')
rouge = evaluate.load('rouge')

eval_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=None,
    padding=True,
)

def hitung_rouge(model, test_dataset):
    model.eval()
    preds = []
    refs  = df_test['target_summary_manual'].tolist()

    test_dataset.set_format('torch')
    loader = DataLoader(test_dataset, batch_size=4, collate_fn=eval_collator)

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            outputs = model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_new_tokens=100,
                no_repeat_ngram_size=3,
                repetition_penalty=2.0,
                num_beams=4,
                early_stopping=True,
            )
            decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            preds.extend(decoded)

    preds = [p.replace(PREFIX, '').strip() for p in preds]

    if preds:
        print(f"  [Prediksi T5]     : {preds[0][:100]}...")
        print(f"  [Referensi Manual]: {refs[0][:100]}...")

    pairs = [(p, r) for p, r in zip(preds, refs) if p.strip()]
    if not pairs:
        return {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0}

    p_clean, r_clean = zip(*pairs)
    scores = rouge.compute(predictions=list(p_clean), references=list(r_clean), use_stemmer=False)
    return {k: round(v, 4) for k, v in scores.items()}


def buat_trainer(model, params, run_dir):
    steps_per_epoch = len(train_ds) // params['per_device_train_batch_size']
    total_steps     = steps_per_epoch * params['num_train_epochs']
    warmup_steps    = max(1, int(total_steps * 0.1))

    args = Seq2SeqTrainingArguments(
        output_dir=str(run_dir),
        num_train_epochs=params['num_train_epochs'],
        per_device_train_batch_size=params['per_device_train_batch_size'],
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=1,
        learning_rate=params['learning_rate'],
        warmup_steps=warmup_steps,
        weight_decay=0.01,
        max_grad_norm=0.5,
        logging_steps=10,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        save_total_limit=1,
        predict_with_generate=True,
        generation_max_length=100,
        fp16=True,
        bf16=False,
        report_to='none',
    )

    trainer_init_params = inspect.signature(Seq2SeqTrainer.__init__).parameters
    tok_kwarg = (
        {'processing_class': tokenizer}
        if 'processing_class' in trainer_init_params
        else {'tokenizer': tokenizer}
    )

    return Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        data_collator=DataCollatorForSeq2Seq(
            tokenizer,
            model=None,          # ← FIX: ganti model= ke None
                                 #   kalau model= diisi, collator generate
                                 #   decoder_input_ids otomatis → konflik dengan labels
            padding=True,
            label_pad_token_id=-100,
        ),
        **tok_kwarg,
    )

def jalankan_satu_run(run_label, params, run_dir, hasil_list, best_state):
    print(f'\n{"="*55}')
    print(f'[{run_label}] Epoch={params["num_train_epochs"]}  BS={params["per_device_train_batch_size"]}  LR={params["learning_rate"]}')
    print(f'{"="*55}')

    model   = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
    trainer = buat_trainer(model, params, run_dir)
    trainer.train()

    scores = hitung_rouge(trainer.model, test_ds)
    print(f'  ROUGE-1={scores["rouge1"]}  ROUGE-2={scores["rouge2"]}  ROUGE-L={scores["rougeL"]}')

    hasil_list.append({
        'run'          : run_label,
        'epoch'        : params['num_train_epochs'],
        'batch_size'   : params['per_device_train_batch_size'],
        'learning_rate': params['learning_rate'],
        'rouge1'       : scores['rouge1'],
        'rouge2'       : scores['rouge2'],
        'rougeL'       : scores['rougeL'],
    })

    if scores['rougeL'] > best_state['rouge_l']:
        best_state['rouge_l'] = scores['rougeL']
        best_state['config']  = params.copy()
        if LOCAL_BEST_DIR.exists():
            shutil.rmtree(LOCAL_BEST_DIR)
        trainer.save_model(str(LOCAL_BEST_DIR))
        tokenizer.save_pretrained(str(LOCAL_BEST_DIR))
        print(f'  ⭐ Model terbaik baru! ROUGE-L={scores["rougeL"]} tersimpan lokal.')

    del model, trainer
    torch.cuda.empty_cache()
    gc.collect()

    if run_dir.exists():
        shutil.rmtree(run_dir)
        print(f'  Folder sementara {run_dir.name} dihapus.')

In [15]:
# ============================================================
# CELL DEBUG — Cek penyebab NaN loss
# ============================================================
import torch

# 1. Cek pad_token_id tokenizer
print(f"pad_token_id : {tokenizer.pad_token_id}")
print(f"eos_token_id : {tokenizer.eos_token_id}")
print()

# 2. Cek sampel label di test_ds
sample = test_ds[0]
labels = sample['labels']
print(f"Label[0] panjang : {len(labels)}")
print(f"Label[0] isi     : {labels[:20]}")
print(f"Ada -100         : {-100 in labels}")
print(f"Ada pad_token    : {tokenizer.pad_token_id in labels}")
print()

# 3. Coba forward pass manual untuk cek loss
from transformers import AutoModelForSeq2SeqLM
from transformers import DataCollatorForSeq2Seq
from torch.utils.data import DataLoader

model_debug = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
collator_debug = DataCollatorForSeq2Seq(
    tokenizer, model=model_debug, padding=True, label_pad_token_id=-100
)

test_ds.set_format('torch')
loader_debug = DataLoader(test_ds.select(range(4)), batch_size=4, collate_fn=collator_debug)
batch = next(iter(loader_debug))

print("Batch keys:", list(batch.keys()))
print("input_ids shape  :", batch['input_ids'].shape)
print("labels shape     :", batch['labels'].shape)
print("Label sample     :", batch['labels'][0][:20])
print("Ada -100 di label:", (-100 in batch['labels'][0]).item())
print()

# 4. Forward pass → cek loss
with torch.no_grad():
    out = model_debug(
        input_ids=batch['input_ids'].to(DEVICE),
        attention_mask=batch['attention_mask'].to(DEVICE),
        labels=batch['labels'].to(DEVICE),
    )
print(f"Loss dari forward pass: {out.loss.item()}")
print(f"Loss NaN? : {torch.isnan(out.loss).item()}")

del model_debug
torch.cuda.empty_cache()

pad_token_id : 0
eos_token_id : 1

Label[0] panjang : 64
Label[0] isi     : tensor([16052,  2639,    43, 17920,     7,  4280,  1693,  3831,  2019,  2347,
          945,   189,   450,  4051, 18387,  1698,   154,  2430, 16962,    11])
Ada -100         : False
Ada pad_token    : False



Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Batch keys: ['input_ids', 'attention_mask', 'labels', 'decoder_input_ids']
input_ids shape  : torch.Size([4, 512])
labels shape     : torch.Size([4, 79])
Label sample     : tensor([16052,  2639,    43, 17920,     7,  4280,  1693,  3831,  2019,  2347,
          945,   189,   450,  4051, 18387,  1698,   154,  2430, 16962,    11])


AttributeError: 'bool' object has no attribute 'item'

## 6. Eksekusi 19 Hyperparameter

In [14]:
import itertools

hasil_eksperimen = []
best_state       = {'rouge_l': -1, 'config': None}
LOCAL_ROOT       = Path('/content/models_temp')
LOCAL_BEST_DIR   = Path('/content/model_juara_lokal')
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

# --- Baseline Zero-Shot ---
print("MENGUJI EXP-BASE: ZERO-SHOT (MODEL TANPA TRAINING)")
model_mentah = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
skor_baseline = hitung_rouge(model_mentah, test_ds)
print(f"  [EXP-BASE] ROUGE-1: {skor_baseline['rouge1']} | ROUGE-2: {skor_baseline['rouge2']} | ROUGE-L: {skor_baseline['rougeL']}")
hasil_eksperimen.append({
    'run': 'EXP-BASE (Zero-Shot)', 'epoch': 0,
    'batch_size': '-', 'learning_rate': '-',
    **skor_baseline
})
del model_mentah
torch.cuda.empty_cache()
gc.collect()

# --- Grid Search Fine-Tuning ---
epochs         = [5, 10, 15]
batch_sizes    = [4, 8]
learning_rates = [1e-5, 5e-5, 3e-5]

kombinasi = list(itertools.product(epochs, batch_sizes, learning_rates))
print(f"\nMEMULAI {len(kombinasi)} EKSPERIMEN FINE-TUNING...")

for i, (ep, bs, lr) in enumerate(kombinasi, 1):
    run_label = f"EXP-{i:03d}"
    run_dir   = LOCAL_ROOT / f"run_{run_label}_ep{ep}_bs{bs}_lr{lr}"
    jalankan_satu_run(
        run_label,
        {'num_train_epochs': ep, 'per_device_train_batch_size': bs, 'learning_rate': lr},
        run_dir,
        hasil_eksperimen,
        best_state,
    )

# --- Rekap Hasil ---
df_hasil = pd.DataFrame(hasil_eksperimen)
print("\nREKAPITULASI HASIL EKSPERIMEN")
display(df_hasil)

df_hasil.to_csv(OUTPUT_ROOT / 'rekap_eksperimen_skripsi.csv', index=False)
print("CSV rekapitulasi tersimpan di Drive!")

# Upload model terbaik ke Drive
print("Mengirim model terbaik ke Google Drive...")
if LOCAL_BEST_DIR.exists():
    if BEST_DIR.exists():
        shutil.rmtree(BEST_DIR)
    shutil.copytree(str(LOCAL_BEST_DIR), str(BEST_DIR))
    print(f"Model terbaik tersimpan di: {BEST_DIR}")
    print(f"Konfigurasi terbaik: {best_state['config']}")
else:
    print("LOCAL_BEST_DIR tidak ditemukan — tidak ada model yang disimpan.")

MENGUJI EXP-BASE: ZERO-SHOT (MODEL TANPA TRAINING)


Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


  [Prediksi T5]     : kami persilakan untuk duduk kembali sidang dewan yang terhormat para anggota dpr dan pimpinan fraksi...
  [Referensi Manual]: Rapat Paripurna DPR RI membahas laporan Komisi III tentang hasil uji kelayakan calon anggota Lembaga...
  [EXP-BASE] ROUGE-1: 0.248 | ROUGE-2: 0.0906 | ROUGE-L: 0.1637

MEMULAI 18 EKSPERIMEN FINE-TUNING...

[EXP-001] Epoch=5  BS=4  LR=1e-05


Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Epoch,Training Loss,Validation Loss
1,0.000000,nan


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 8. Visualisasi Hasil

In [ ]:
import matplotlib.pyplot as plt

df_hasil_sorted = df_hasil.sort_values('rougeL', ascending=False).reset_index(drop=True)

# Grafik 1: ROUGE-L per Run
fig, ax = plt.subplots(figsize=(14, 6))
colors = ['#E74C3C' if 'Zero-Shot' in str(r) else '#4A90D9' for r in df_hasil_sorted['run']]
bars = ax.bar(df_hasil_sorted['run'], df_hasil_sorted['rougeL'], color=colors)
ax.set_title('ROUGE-L per Eksperimen (Merah = Zero-Shot Baseline)', fontsize=14, fontweight='bold')
ax.set_xlabel('Skenario / Run', fontsize=12)
ax.set_ylabel('Skor ROUGE-L', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=9)
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 0.005,
            round(yval, 4), ha='center', va='bottom', fontsize=9, rotation=90)
ax.set_ylim(0, max(df_hasil_sorted['rougeL']) * 1.3)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'rouge_per_run.png', dpi=150, bbox_inches='tight')
plt.show()

# Grafik 2: Pengaruh Hyperparameter
df_grid = df_hasil[df_hasil['run'] != 'EXP-BASE (Zero-Shot)'].copy()
df_grid['epoch'] = df_grid['epoch'].astype(int)
df_grid['batch_size'] = df_grid['batch_size'].astype(int)

fig2, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
fig2.suptitle('Pengaruh Hyperparameter terhadap Rata-Rata ROUGE-L', fontsize=14, fontweight='bold', y=1.05)

for ax, col, label in zip(axes, ['epoch', 'batch_size', 'learning_rate'], ['Epoch', 'Batch Size', 'Learning Rate']):
    g = df_grid.groupby(col)['rougeL'].mean().reset_index()
    bars_hp = ax.bar(g[col].astype(str), g['rougeL'], color='#2ECC71', edgecolor='black')
    ax.set_title(f'ROUGE-L vs {label}', fontsize=12)
    ax.set_xlabel(label, fontsize=11)
    if ax == axes[0]:
        ax.set_ylabel('Rata-rata ROUGE-L', fontsize=11)
    for bar in bars_hp:
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, yval + 0.002,
                round(yval, 4), ha='center', va='bottom', fontsize=10)
axes[0].set_ylim(0, max(df_grid['rougeL']) * 1.2)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'visualisasi_hyperparameter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik visualisasi sudah siap!')

##Evaluasi Final Model Terbaik

In [ ]:
from torch.utils.data import DataLoader

print("EVALUASI FINAL: MODEL TERBAIK SETELAH TUNING")
print(f"Memuat dari: {BEST_DIR}...")

model_juara    = AutoModelForSeq2SeqLM.from_pretrained(BEST_DIR).to(DEVICE)
tokenizer_juara = AutoTokenizer.from_pretrained(BEST_DIR)
model_juara.eval()

preds_final = []
refs_final  = df_test['target_summary_manual'].tolist()

test_ds.set_format('torch')
loader_final = DataLoader(test_ds, batch_size=4)

print("Sedang memproses evaluasi akhir...")
with torch.no_grad():
    for batch in loader_final:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        outputs = model_juara.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=100,
            no_repeat_ngram_size=3,
            num_beams=4,
            repetition_penalty=2.0,
            early_stopping=True,
        )
        decoded = tokenizer_juara.batch_decode(outputs, skip_special_tokens=True)
        preds_final.extend(decoded)

preds_final = [p.replace(PREFIX, '').strip() for p in preds_final]
pairs = [(p, r) for p, r in zip(preds_final, refs_final) if p.strip()]

if pairs:
    p_clean, r_clean = zip(*pairs)
    rouge_final = evaluate.load('rouge')
    scores_final = rouge_final.compute(
        predictions=list(p_clean),
        references=list(r_clean),
        use_stemmer=False
    )
    print("\n=======================================================")
    print("SKOR ROUGE FINAL MODEL TERBAIK")
    print("=======================================================")
    print(f"  ROUGE-1 : {round(scores_final['rouge1'], 4)}")
    print(f"  ROUGE-2 : {round(scores_final['rouge2'], 4)}")
    print(f"  ROUGE-L : {round(scores_final['rougeL'], 4)}")
    print("\nBANDINGKAN HASIL TEKS:")
    print(f"  [Ringkasan Manual Asli] : {r_clean[0][:200]}...")
    print(f"  [Prediksi Model IndoT5] : {p_clean[0][:200]}...")
else:
    print("Gagal menghitung ROUGE.")

del model_juara
torch.cuda.empty_cache()
gc.collect()